# Origem (MongoDB) → Landing

Lê as coleções do MongoDB Atlas e grava cada uma como arquivo JSON no bucket `landing` do MinIO.

**Formato Landing:** JSON bruto, sem transformações — conforme requisito do trabalho (NoSQL → JSON).

> `minio` já está disponível via startup script.

## 1. Conexão com o MongoDB Atlas

In [6]:
import os
import json
import io
from datetime import datetime, date
from bson import ObjectId
from pymongo import MongoClient

MONGO_URI  = os.environ["MONGO_URI"]
MONGO_DB   = os.environ["MONGO_DB_NAME"]
BUCKET     = "landing"
PREFIX     = "tse"

client = MongoClient(MONGO_URI)
db     = client[MONGO_DB]

colecoes = db.list_collection_names()
print(f"Banco    : {MONGO_DB}")
print(f"Coleções : {len(colecoes)}")
for c in sorted(colecoes):
    print(f"  - {c}")

Banco    : trabalho-final-engenharia-dados
Coleções : 10
  - bens
  - bens_estatistica
  - candidatura
  - cargo
  - instrucao
  - municipio
  - ocupacao
  - partido
  - situacao
  - tipo_bem


## 2. Função de ingestão

In [3]:
class MongoEncoder(json.JSONEncoder):
    """Serializa tipos do MongoDB que o json padrão não suporta."""
    def default(self, obj):
        if isinstance(obj, ObjectId):
            return str(obj)
        if isinstance(obj, (datetime, date)):
            return obj.isoformat()
        return super().default(obj)


def colecao_para_landing(nome: str) -> int:
    """
    Lê uma coleção do MongoDB e grava como JSON no bucket landing.

    Parâmetros
    ----------
    nome : str
        Nome da coleção no MongoDB.

    Retorna
    -------
    int
        Quantidade de documentos gravados.
    """
    documentos = list(db[nome].find())
    total = len(documentos)

    conteudo = json.dumps(documentos, cls=MongoEncoder, ensure_ascii=False, indent=2)
    dados    = io.BytesIO(conteudo.encode("utf-8"))
    chave    = f"{PREFIX}/{nome}.json"

    minio.put_object(
        Bucket=BUCKET,
        Key=chave,
        Body=dados,
        ContentLength=len(conteudo.encode("utf-8")),
        ContentType="application/json",
    )

    print(f"  [{nome}] {total:,} documentos → s3://{BUCKET}/{chave}")
    return total

## 3. Ingestão de todas as coleções

In [4]:
print(f"Iniciando ingestão: {len(colecoes)} coleção(ões)...\n")

totais = {}
for nome in sorted(colecoes):
    totais[nome] = colecao_para_landing(nome)

print("\nIngestão concluída.")

Iniciando ingestão: 10 coleção(ões)...

  [bens] 157,803 documentos → s3://landing/tse/bens.json
  [bens_estatistica] 49,294 documentos → s3://landing/tse/bens_estatistica.json
  [candidatura] 78,349 documentos → s3://landing/tse/candidatura.json
  [cargo] 3 documentos → s3://landing/tse/cargo.json
  [instrucao] 8 documentos → s3://landing/tse/instrucao.json
  [municipio] 645 documentos → s3://landing/tse/municipio.json
  [ocupacao] 241 documentos → s3://landing/tse/ocupacao.json
  [partido] 30 documentos → s3://landing/tse/partido.json
  [situacao] 2,685 documentos → s3://landing/tse/situacao.json
  [tipo_bem] 50 documentos → s3://landing/tse/tipo_bem.json

Ingestão concluída.


## 4. Validação da camada Landing

In [5]:
print("=== Validação Landing ===")
print(f"{'Coleção':<30} {'Documentos':>12} {'Tamanho':>12}")
print("-" * 56)

total_geral = 0
resp = minio.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)

for obj in sorted(resp.get("Contents", []), key=lambda x: x["Key"]):
    nome_col = obj["Key"].replace(f"{PREFIX}/", "").replace(".json", "")
    docs = totais.get(nome_col, 0)
    tamanho  = f"{obj['Size'] / 1024:.1f} KB"
    total_geral += docs if isinstance(docs, int) else 0
    print(f"{nome_col:<30} {docs:>12,} {tamanho:>12}")

print("-" * 56)
print(f"{'TOTAL':<30} {total_geral:>12,}")

client.close()

=== Validação Landing ===
Coleção                          Documentos      Tamanho
--------------------------------------------------------


ValueError: Cannot specify ',' with 's'.